In [1]:
!ls stand

attempt			 games_dict		 meta.json
bin			 games_flog.gz		 order_money_left_caesar
config			 games_flog.gz.PREV	 poor_mans_profiler
daemon.json		 geodata4-tree+ling.bin  push-client
dssm			 geodata6.bin		 secret
dumped_requests.gz	 language		 service_tickets
dumped_requests.gz.PREV  lm			 stand.json
formulas		 log.local.txt		 unified_agent


In [2]:
!head stand/log.local.txt -

Model = 6; [-0.211147,0.028769,-0.242279,0.0841654,0.0248732,0.0154993,0.0717655,0.0942284,-0.00700669,-0.0135621,0.0214947,0.022302,0.252389,-0.0336773,-0.138279,-0.0534052,0.0690308,-0.197792,-0.109553,-0.00141407,0.0443739,0.172978,-0.00064369,0.0590741,-0.0824093,-0.0682868,0.0344061,0.0911272,-0.0051644,-0.0839934,0.00860636,-0.0956932,-0.0486178,-0.0878616,-0.00121456,-0.0839231,-0.0282804,0.00692664,0.102235,-0.0945008,0.0576525,0.0920146,0.140158,-0.0620869,-0.20815,0.00623474,0.105515,0.215129,0.0282423,-0.0290657,-0.0268382,0.0365566,-0.0845465,0.0747499,-0.0457398,-0.0335554,-0.0625323,-0.11611,-0.00558053,0.0719073,-0.114699,0.0206136,-0.0253669,-0.0439999,0.326458,-0.0629797,-0.0288919,-0.0863563,-0.0421662,-0.0153366,-0.0748693,-0.0997181,-0.0855427,-0.105331,0.0707908,0.0398544,0.0028728,0.155376,0.138392,-0.00578193,-0.00382951,0.0407524,0.090671,0.267034,0.0891853,0.00231596,0.159994,0.105253,0.149872,0.00413346,0.0340003,-0.109571,0.0123925,0.0624056,-0.0539893,-0.031

In [3]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [127]:
import collections
import numpy as np
import tqdm

readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
requests = list()
docembs = collections.defaultdict(dict)
with open("stand/log.local.txt") as f:
    req = list()
    reqid = None
    models = list()
    prevreqmodel = None
    reqmodel = dict()
    prevmodelid = -1
    bannermodelid = -1
    for i, line in tqdm.tqdm_notebook(enumerate(f)):
        if line.startswith("Model = 6;"):
            prevreqmodel = reqmodel
            reqmodel = dict()
        
        if line.startswith("dbid 184394"):
            print(line),
            print(prevmodelid)
            
        if line.startswith("Model = "):
            spl = line.split(" ")
            prevmodelid = int(spl[2][:-1])
            bannermodelid = max(bannermodelid , prevmodelid)
            reqmodel[prevmodelid] = readvector(spl[3])
        elif line.startswith("dbid"):
            spl = line.split(" ")
            dbid = int(spl[1][:-1])
            docembs[bannermodelid][dbid] = readvector(spl[2])
        elif line.startswith("seed"):
            if len(requests) >= 1000:
                break
            if req:
                requests.append((reqid, prevreqmodel, sorted(req)))
                req = list()
            reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
        else:
            req.append(
                (int(line.split()[0]), float(line.split()[1]))
            )

requests = [r for r in requests if len(r[2]) == 16514]

dbid 184394; [-0.0512872,0.0193015,0.113894,0.0252779,-0.155716,0.123881,-0.0206214,0.0627056,-0.0278315,0.372551,0.000806784,-0.0653532,-0.0687121,0.0878991,0.0476131,0.140274,-0.170451,0.0202663,0.141594,-0.066827,0.0701703,-0.130011,-0.0651865,0.0546216,0.152157,0.229269,-0.126042,0.0172359,-0.0209199,0.0127505,-0.122559,-0.0714047,0.062702,0.0793581,0.0914689,0.207112,0.179825,0.0887367,-0.168106,0.0249156,-0.119067,-0.0720207,-0.0911074,-0.0435966,0.028035,0.0111506,0.0389887,-0.112137,-0.0404294,-0.0934756,0.0164074,-0.0955409,0.00808596,-0.0411514,-0.0539788,0.00297383,0.0199476,0.134285,0.0632445,-0.0619632,-0.110289,0.0151936,0.135167,0.105939,-0.079014,-0.0997647,0.000153098,-0.0108563,0.0231189,0.0272427,-0.0255719,-0.0711633,-0.0625287,0.0397355,0.0102844,-0.149684,0.0828743,0.142888,-0.0176144,-0.0116712,0.0371071,-0.212492,-0.0232992,-0.0258352,-0.0230406,-0.0762529,-0.0378617,-0.041919,-0.091265,-0.0543666,-0.0468707,0.00201314,0.128076,-0.284469,0.0124104,0.0697693,0.05

In [131]:
len(docembs[6].keys()), len(docembs[9].keys()), 

(16514, 16514)

In [132]:
docblocks = {
    mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
    for mid in docembs
}
docblocks

{6: array([[ 0.0407761 , -0.0866397 ,  0.0973855 , ..., -0.0337863 ,
          0.1449    ,  0.023338  ],
        [-0.0173765 ,  0.103833  ,  0.0541691 , ...,  0.0947965 ,
          0.119532  ,  0.111093  ],
        [-0.010643  , -0.0188779 ,  0.128833  , ..., -0.0185986 ,
          0.0270978 ,  0.0370004 ],
        ...,
        [-0.0822528 ,  0.0734386 ,  0.0820323 , ..., -0.0283938 ,
          0.0727261 ,  0.244681  ],
        [-0.0396752 ,  0.215461  ,  0.148155  , ..., -0.0594413 ,
          0.0757048 ,  0.140465  ],
        [-0.0272712 ,  0.207217  ,  0.154168  , ..., -0.00856583,
          0.0459061 ,  0.18192   ]]),
 7: array([[-0.0457229 ,  0.0874625 ,  0.0102194 , ...,  0.0135331 ,
         -0.0444551 ,  0.012255  ],
        [-0.135033  ,  0.0949462 , -0.0646434 , ...,  0.105708  ,
         -0.0441772 ,  0.0313092 ],
        [-0.176407  ,  0.15634   , -0.0423812 , ...,  0.165932  ,
         -0.0977578 , -0.0507538 ],
        ...,
        [ 0.0745855 ,  0.15481   , -0.0633103 , 

In [129]:
(requests[0][1][6] ** 2).sum()

0.9994653378993866

In [8]:
games_count = 16514
key_games = np.random.choice(np.arange(games_count), 100, replace=False)
key_reqs = np.random.choice(np.arange(len(requests)), 101, replace=False)
key_games, key_reqs

(array([ 6614, 14966, 10690,  8533, 15779, 14625, 11219, 10536,  6360,
         4518, 10767,  6664, 14121, 13467,  9488,  9525,  9701, 11426,
         5683,  2785,  2135,  6182,  8929,  3685,  8397,  3406,  6473,
         9673, 10547, 14024,   286,   605,  8571, 15067,  9924,  1511,
         5108,  8877, 11987, 11278, 13728,  8757,  9343, 12229,  7191,
         7082,  4420,  4899, 14129,  4551,  2811,  1724,  7570,  1982,
         1880, 10204,  4393,  9403,  1470, 10867,  6541,  9236, 14475,
         8636, 11843, 11291,  8313, 16111,  4801,  2894, 13153, 10209,
         8148,   838, 10619,  3235,    47, 10577, 12966,  4703,   651,
        14780, 13298,  9781,  1664,  8161,  2515,  3309,  4092, 10711,
        12028,  8283,  2266, 12228, 12884,  5320, 13123,   565,  7488,
         8502]),
 array([583, 898, 956, 423, 680, 301, 723, 532, 791, 670, 897, 367, 576,
         33, 360, 730, 330, 663, 714, 217,  66, 905, 792, 564, 172, 539,
        116, 677, 781, 809, 911, 122, 385, 674, 701, 780

In [125]:
len(requests)

987

In [130]:
requests[0][1]

{6: array([-0.211147  ,  0.028769  , -0.242279  ,  0.0841654 ,  0.0248732 ,
         0.0154993 ,  0.0717655 ,  0.0942284 , -0.00700669, -0.0135621 ,
         0.0214947 ,  0.022302  ,  0.252389  , -0.0336773 , -0.138279  ,
        -0.0534052 ,  0.0690308 , -0.197792  , -0.109553  , -0.00141407,
         0.0443739 ,  0.172978  , -0.00064369,  0.0590741 , -0.0824093 ,
        -0.0682868 ,  0.0344061 ,  0.0911272 , -0.0051644 , -0.0839934 ,
         0.00860636, -0.0956932 , -0.0486178 , -0.0878616 , -0.00121456,
        -0.0839231 , -0.0282804 ,  0.00692664,  0.102235  , -0.0945008 ,
         0.0576525 ,  0.0920146 ,  0.140158  , -0.0620869 , -0.20815   ,
         0.00623474,  0.105515  ,  0.215129  ,  0.0282423 , -0.0290657 ,
        -0.0268382 ,  0.0365566 , -0.0845465 ,  0.0747499 , -0.0457398 ,
        -0.0335554 , -0.0625323 , -0.11611   , -0.00558053,  0.0719073 ,
        -0.114699  ,  0.0206136 , -0.0253669 , -0.0439999 ,  0.326458  ,
        -0.0629797 , -0.0288919 , -0.0863563 , -

In [11]:
embed_users = np.array([
    np.array([r_i[2][i][1] for i in key_games])
    for r_i in requests
])
embed_users_mean = embed_users.mean(axis = 0)
# embed_users = embed_users - embed_users_mean
embed_users.shape

(987, 100)

In [12]:
embed_games = np.array([
    np.array([requests[r_i][2][g_i][1] for r_i in key_reqs])
    for g_i in range(games_count)
])
embed_games_mean = embed_games.mean(axis = 0)
# embed_games = embed_games - embed_games_mean
embed_games.shape

(16514, 101)

In [13]:
games_top = embed_games.mean(axis = 1)

In [14]:
games2users = np.array([
    embed_games[g_i]
    for g_i in key_games
])
games2users.shape

(100, 101)

In [15]:
core_users_scores = np.array([
    np.array([g_i[1] for g_i in requests[r_i][2]])
    for r_i in key_reqs
])
core_users_scores.shape

(101, 16514)

In [16]:
embed_users.shape

(987, 100)

In [17]:
core_users_embs = embed_users[key_reqs]
core_users_embs.shape

(101, 100)

In [18]:
ge_users = (embed_users.T / embed_users.mean(axis=1)).T @ games2users
# ge_users = embed_users @ games2users
ge_users.shape

(987, 101)

In [19]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(500):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 5000
        )
        print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)
1.3809978
1.3565859
1.3446248
1.3367627
1.3296856
1.3231108
1.3174672
1.3128494
1.3089887
1.3055356
1.3022738
1.2991562
1.2962204
1.293509
1.2910264
1.2887428
1.2866224
1.2846495
1.2828252
1.2811533
1.2796245
1.2782174
1.2769054
1.2756666
1.2744875
1.2733603
1.2722818
1.2712518
1.270272
1.269342
1.2684588
1.2676183
1.2668152
1.2660488
1.2653213
1.2646358
1.2639906
1.2633798
1.2627935
1.2622273
1.2616822
1.2611645
1.260676
1.2602128
1.2597687
1.2593403
1.2589296
1.2585392
1.2581708
1.2578216
1.257487
1.2571647
1.2568544
1.2565582
1.256277
1.2560085
1.2557504
1.2555016
1.2552637
1.2550373
1.2548212
1.254615
1.2544166
1.2542267
1.2540458
1.2538736
1.2537091
1.2535521
1.2534018
1.2532586
1.253123
1.2529937
1.2528707
1.2527536
1.2526419
1.2525364
1.252436
1.2523407
1.2522503
1.2521644
1.252083
1.2520057
1.2519327
1.2518632
1.2517978
1.2517357
1.2516768
1.251621
1.2515686
1.2515188
1.2514715
1.2514272
1.2513853
1.2513454
1.25130

In [20]:
tf.reduce_mean(W * W) * 1

<tf.Tensor: shape=(), dtype=float32, numpy=6.517239e-06>

In [21]:
from scipy.spatial.distance import cosine

from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

from sklearn.metrics import pairwise 

In [22]:
dir(pairwise)

['DataConversionWarning',
 'KERNEL_PARAMS',
 'PAIRED_DISTANCES',
 'PAIRWISE_BOOLEAN_FUNCTIONS',
 'PAIRWISE_DISTANCE_FUNCTIONS',
 'PAIRWISE_KERNEL_FUNCTIONS',
 'Parallel',
 '_NAN_METRICS',
 '_VALID_METRICS',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_argmin_min_reduce',
 '_check_chunk_size',
 '_chi2_kernel_fast',
 '_deprecate_positional_args',
 '_dist_wrapper',
 '_euclidean_distances_upcast',
 '_get_mask',
 '_num_samples',
 '_pairwise_callable',
 '_parallel_pairwise',
 '_precompute_metric_params',
 '_return_float_dtype',
 '_sparse_manhattan',
 'additive_chi2_kernel',
 'check_array',
 'check_non_negative',
 'check_paired_arrays',
 'check_pairwise_arrays',
 'chi2_kernel',
 'cosine_distances',
 'cosine_similarity',
 'csr_matrix',
 'delayed',
 'distance',
 'distance_metrics',
 'effective_n_jobs',
 'euclidean_distances',
 'gen_batches',
 'gen_even_slices',
 'get_chunk_n_rows',
 'haversine_distances',
 'is_scalar_nan',


In [23]:
import matplotlib.pyplot as plt
plt.hist(embed_games.flatten())

(array([1.667877e+06, 2.000000e+01, 6.000000e+00, 4.000000e+00,
        1.000000e+00, 4.000000e+00, 0.000000e+00, 0.000000e+00,
        1.000000e+00, 1.000000e+00]),
 array([9.74706709e-05, 6.56539696e+01, 1.31307842e+02, 1.96961714e+02,
        2.62615586e+02, 3.28269458e+02, 3.93923330e+02, 4.59577202e+02,
        5.25231074e+02, 5.90884946e+02, 6.56538818e+02]),
 <a list of 10 Patch objects>)

In [25]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.005857787810383747 0.4473702031602709
d -> 0.4814559819413093 0.006038374717832957 0.4473702031602709
c -> 0.01746049661399549 0.005959367945823928 0.4473702031602709
w -> 0.4944808126410835 0.0060835214446952595 0.4473702031602709


In [26]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.003182844243792325 0.3336343115124153
d -> 0.36880361173814896 0.0031828442437923255 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.38158013544018066 0.003069977426636569 0.3336343115124153


Тюним

In [135]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 1000
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)
1.3766173
1.2021382
1.1933919
1.1919646
1.1917517
1.1917268
1.1917244
1.1917244
1.1917244
1.1917243


In [28]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3762076749435666 0.002618510158013544 0.3336343115124153
d -> 0.36880361173814896 0.0029119638826185104 0.3336343115124153
c -> 0.008577878103837472 0.002957110609480813 0.3336343115124153
w -> 0.40388261851015805 0.0027539503386004517 0.3336343115124153


In [29]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.4731151241534989 0.006422121896162527 0.4473702031602709
d -> 0.4814559819413093 0.0060609480812641075 0.4473702031602709
c -> 0.01746049661399549 0.005711060948081265 0.4473702031602709
w -> 0.512020316027088 0.005970654627539504 0.4473702031602709


# dssm?

In [136]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-docblocks[i] @ requests_i[1][i] for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.15483069977426636 0.0031602708803611735 0.3336343115124153
7 -> 0.11663656884875846 0.0034085778781038373 0.3336343115124153
8 -> 0.15386004514672688 0.0032054176072234763 0.3336343115124153
9 -> 0.0362979683972912 0.0032054176072234763 0.3336343115124153
e -> 0.3762076749435666 0.0031602708803611735 0.3336343115124153
d -> 0.36880361173814896 0.0028442437923250565 0.3336343115124153
c -> 0.008577878103837472 0.0029119638826185104 0.3336343115124153
w -> 0.40388261851015805 0.002618510158013544 0.3336343115124153


In [137]:
games_top

array([0.0047713 , 0.00415357, 0.00380251, ..., 0.1423562 , 0.04750573,
       0.07614477])

In [141]:
dssmid = 7
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
7 -> 0.3336343115124153 0.0031602708803611735 0.3336343115124153
x =  1
7 -> 0.35379232505643343 0.0027765237020316025 0.3336343115124153
x =  2
7 -> 0.37090293453724604 0.0031602708803611735 0.3336343115124153
x =  3
7 -> 0.38564334085778784 0.002889390519187359 0.3336343115124153
x =  4
7 -> 0.40259593679458244 0.002663656884875847 0.3336343115124153
x =  5
7 -> 0.4182844243792325 0.002641083521444696 0.3336343115124153
x =  6
7 -> 0.4323702031602709 0.0031602708803611743 0.3336343115124153
x =  7
7 -> 0.4396839729119639 0.0028668171557562076 0.3336343115124153
x =  8
7 -> 0.44162528216704294 0.0034085778781038373 0.3336343115124153
x =  9
7 -> 0.4411286681715576 0.0035665914221218965 0.3336343115124153
x =  10
7 -> 0.4392099322799097 0.0030248306997742664 0.3336343115124153
x =  11
7 -> 0.43462753950338606 0.0030248306997742664 0.3336343115124153
x =  12
7 -> 0.4294356659142212 0.003386004514672686 0.3336343115124153
x =  13
7 -> 0.42151241534988715 0.003363431151241535 0.333

In [142]:
dssmid = 6
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.3336343115124153 0.0030925507900677203 0.3336343115124153
x =  1
6 -> 0.37266365688487585 0.003002257336343115 0.3336343115124153
x =  2
6 -> 0.3972911963882618 0.003340857787810384 0.3336343115124153
x =  3
6 -> 0.42458239277652365 0.0024830699774266367 0.3336343115124153
x =  4
6 -> 0.45449209932279916 0.003115124153498871 0.3336343115124153
x =  5
6 -> 0.4766591422121897 0.0030699774266365687 0.3336343115124153
x =  6
6 -> 0.4895033860045147 0.0032054176072234763 0.3336343115124153
x =  7
6 -> 0.4941083521444696 0.0032731376975169302 0.3336343115124153
x =  8
6 -> 0.49146726862302487 0.003002257336343115 0.3336343115124153
x =  9
6 -> 0.48632054176072237 0.002595936794582393 0.3336343115124153
x =  10
6 -> 0.4764559819413092 0.0028893905191873593 0.3336343115124153
x =  11
6 -> 0.4654853273137698 0.0027313769751693 0.3336343115124153
x =  12
6 -> 0.45309255079006777 0.0030248306997742672 0.3336343115124153
x =  13
6 -> 0.43932279909706545 0.003024830699774266 0.3336343

In [152]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }

    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.3336343115124153 0.0029345372460496616 0.3336343115124153
x =  1
9 -> 0.34738148984198647 0.002844243792325057 0.3336343115124153
x =  2
9 -> 0.3593227990970655 0.003002257336343115 0.3336343115124153
x =  3
9 -> 0.3699097065462754 0.0030248306997742672 0.3336343115124153
x =  4
9 -> 0.37024830699774264 0.003024830699774266 0.3336343115124153
x =  5
9 -> 0.3647178329571106 0.002595936794582393 0.3336343115124153
x =  6
9 -> 0.3520767494356659 0.0028668171557562076 0.3336343115124153
x =  7
9 -> 0.3291196388261851 0.003047404063205418 0.3336343115124153
x =  8
9 -> 0.30002257336343113 0.002641083521444696 0.3336343115124153
x =  9
9 -> 0.2689390519187359 0.002889390519187359 0.3336343115124153
x =  10
9 -> 0.24376975169300227 0.0027313769751693 0.3336343115124153
x =  11
9 -> 0.22112866817155757 0.0031828442437923255 0.3336343115124153
x =  12
9 -> 0.20067720090293456 0.0034085778781038373 0.3336343115124153
x =  13
9 -> 0.18354401805869075 0.0030925507900677203 0.33363431

In [143]:
for x in range(20):
    print("x = ", x)
    rank_w = np.argsort(-x*embed_users @ W @ embed_games.T-games_top, axis=1)


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_w, "w"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
w -> 0.3336343115124153 0.0032279909706546274 0.3336343115124153
x =  1
w -> 0.39647855530474035 0.0033182844243792326 0.3336343115124153
x =  2
w -> 0.40907449209932284 0.0030925507900677203 0.3336343115124153
x =  3
w -> 0.4110835214446953 0.0031828442437923255 0.3336343115124153
x =  4
w -> 0.41049661399548537 0.0035214446952595937 0.3336343115124153
x =  5
w -> 0.4100451467268624 0.0034085778781038373 0.3336343115124153
x =  6
w -> 0.4096839729119639 0.003340857787810384 0.3336343115124153
x =  7
w -> 0.40936794582392777 0.0034762979683972913 0.3336343115124153
x =  8
w -> 0.40880361173814905 0.0031602708803611735 0.3336343115124153
x =  9
w -> 0.4087810383747179 0.0030248306997742664 0.3336343115124153
x =  10
w -> 0.4089164785553048 0.002934537246049662 0.3336343115124153
x =  11
w -> 0.4089164785553048 0.0027313769751693 0.3336343115124153
x =  12
w -> 0.40887133182844243 0.003295711060948081 0.3336343115124153
x =  13
w -> 0.4086004514672686 0.002595936794582393 0.333634

In [147]:
print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
0-loss =  tf.Tensor(1.375386919419621, shape=(), dtype=float64)


In [148]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T-games_top) ** 2)
            + tf.reduce_mean(W * W) * 150
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.3186524117017808, shape=(), dtype=float64)
1.3189981
1.1485099
1.1218199
1.110669
1.1049802
1.1020124
1.1004989
1.0997444
1.0993747
1.0991968


In [149]:
rank_w = np.argsort(-embed_users @ W @ embed_games.T-games_top, axis=1)


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_w, "w"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids

            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

w -> 0.4295936794582393 0.002595936794582393 0.3336343115124153


In [41]:
500 -> .4108
250 -> .4117
150 -> .4143
100 -> .4151
050 -> .4142
010 -> .4115
000 -> .4033

SyntaxError: invalid syntax (<ipython-input-41-1b4e77c0b5a7>, line 1)

In [150]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.4941083521444696 0.0028216704288939053 0.3336343115124153
7 -> 0.4396839729119639 0.0032054176072234763 0.3336343115124153
8 -> 0.47376975169300223 0.0033182844243792326 0.3336343115124153
9 -> 0.3291196388261851 0.0028668171557562076 0.3336343115124153
e -> 0.3762076749435666 0.0031376975169300227 0.3336343115124153
d -> 0.36880361173814896 0.0027539503386004517 0.3336343115124153
c -> 0.008577878103837472 0.002799097065462754 0.3336343115124153
w -> 0.4295936794582393 0.0032054176072234763 0.3336343115124153


In [153]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-7 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5568735891647856 0.006151241534988712 0.4473702031602709
7 -> 0.5192212189616252 0.005846501128668171 0.4473702031602709
8 -> 0.567178329571106 0.006455981941309256 0.4473702031602709
9 -> 0.37682844243792324 0.006264108352144469 0.4473702031602709
e -> 0.4731151241534989 0.006218961625282167 0.4473702031602709
d -> 0.4814559819413093 0.006060948081264108 0.4473702031602709
c -> 0.01746049661399549 0.005812641083521445 0.4473702031602709
w -> 0.527155756207675 0.006060948081264108 0.4473702031602709


In [154]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-5 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.5759367945823928 0.005711060948081263 0.4473702031602709
7 -> 0.5242776523702032 0.006015801354401806 0.4473702031602709
8 -> 0.5645372460496614 0.005744920993227991 0.4473702031602709
9 -> 0.44339729119638827 0.006015801354401806 0.4473702031602709
e -> 0.4731151241534989 0.005970654627539504 0.4473702031602709
d -> 0.4814559819413093 0.006049661399548532 0.4473702031602709
c -> 0.01746049661399549 0.006309255079006772 0.4473702031602709
w -> 0.527155756207675 0.006230248306997742 0.4473702031602709


In [160]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 200
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.666410835214447 0.01167607223476298 0.5744469525959367
7 -> 0.6234255079006772 0.011834085778781037 0.5744469525959367
8 -> 0.6523137697516931 0.012178329571106095 0.5744469525959367
9 -> 0.5699266365688487 0.011811512415349886 0.5744469525959367
e -> 0.5722686230248306 0.01195823927765237 0.5744469525959367
d -> 0.5961343115124152 0.012076749435665914 0.5744469525959367
c -> 0.030299097065462757 0.012753950338600452 0.5744469525959367
w -> 0.626089164785553 0.011726862302483071 0.5744469525959367


In [161]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T - games_top, axis=1)
rank_di = {
    i: np.argsort([-3 * docblocks[i] @ requests_i[1][i] - games_top for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 250
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]


for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(real_).intersection(set(rec_)))) / float(len(real_))

            rec_res.append(ev(rec, real))
            rand_res.append(ev(random, real))
            top_res.append(ev(top, real))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.6831467268623025 0.014785553047404065 0.5979503386004515
7 -> 0.6498690744920992 0.015480812641083523 0.5979503386004515
8 -> 0.6795124153498872 0.01450112866817156 0.5979503386004515
9 -> 0.5858916478555304 0.014961625282167042 0.5979503386004515
e -> 0.592334085778781 0.015431151241534989 0.5979503386004515
d -> 0.6212821670428893 0.015327313769751695 0.5979503386004515
c -> 0.03572460496613995 0.015449209932279913 0.5979503386004515
w -> 0.6479593679458239 0.015462753950338602 0.5979503386004515


In [51]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.5744469525959367 0.012121896162528218 0.5744469525959367
x =  1
6 -> 0.6088205417607224 0.01252257336343115 0.5744469525959367
x =  2
6 -> 0.6479627539503386 0.01204288939051919 0.5744469525959367
x =  3
6 -> 0.664283295711061 0.012088036117381492 0.5744469525959367
x =  4
6 -> 0.6479740406320542 0.011580135440180587 0.5744469525959367
x =  5
6 -> 0.6167832957110609 0.011839729119638827 0.5744469525959367
x =  6
6 -> 0.5822968397291196 0.012025959367945822 0.5744469525959367
x =  7
6 -> 0.5485609480812641 0.012291196388261852 0.5744469525959367
x =  8
6 -> 0.5183069977426636 0.012127539503386006 0.5744469525959367
x =  9
6 -> 0.49157449209932275 0.011930022573363432 0.5744469525959367
x =  10
6 -> 0.46870203160270885 0.012020316027088036 0.5744469525959367
x =  11
6 -> 0.44742099322799095 0.012477426636568848 0.5744469525959367
x =  12
6 -> 0.4283465011286682 0.012116252821670429 0.5744469525959367
x =  13
6 -> 0.41200338600451464 0.012003386004514673 0.5744469525959367
x

In [52]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.4473702031602709 0.0059480812641083515 0.4473702031602709
x =  1
6 -> 0.4830135440180587 0.006241534988713319 0.4473702031602709
x =  2
6 -> 0.5132167042889391 0.006478555304740407 0.4473702031602709
x =  3
6 -> 0.5453160270880361 0.006038374717832957 0.4473702031602709
x =  4
6 -> 0.56813769751693 0.006207674943566591 0.4473702031602709
x =  5
6 -> 0.5727878103837472 0.006015801354401806 0.4473702031602709
x =  6
6 -> 0.5652934537246049 0.005857787810383747 0.4473702031602709
x =  7
6 -> 0.5506546275395033 0.005575620767494356 0.4473702031602709
x =  8
6 -> 0.5308352144469526 0.005744920993227991 0.4473702031602709
x =  9
6 -> 0.5122234762979685 0.006106094808126411 0.4473702031602709
x =  10
6 -> 0.4936230248306998 0.006591422121896162 0.4473702031602709
x =  11
6 -> 0.47469525959367953 0.00618510158013544 0.4473702031602709
x =  12
6 -> 0.45573363431151237 0.006218961625282167 0.4473702031602709
x =  13
6 -> 0.43795711060948084 0.0059480812641083515 0.4473702031602709


In [158]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 100
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.4473702031602709 0.0063318284424379225 0.4473702031602709
x =  1
9 -> 0.45654627539503384 0.005846501128668171 0.4473702031602709
x =  2
9 -> 0.4679345372460497 0.006331828442437923 0.4473702031602709
x =  3
9 -> 0.4737810383747178 0.0060270880361173815 0.4473702031602709
x =  4
9 -> 0.46619638826185095 0.006218961625282167 0.4473702031602709
x =  5
9 -> 0.44339729119638827 0.006399548532731377 0.4473702031602709
x =  6
9 -> 0.4103386004514672 0.006399548532731377 0.4473702031602709
x =  7
9 -> 0.37682844243792324 0.005688487584650113 0.4473702031602709
x =  8
9 -> 0.3405079006772009 0.005654627539503386 0.4473702031602709
x =  9
9 -> 0.30753950338600455 0.0060270880361173815 0.4473702031602709
x =  10
9 -> 0.27742663656884875 0.006309255079006772 0.4473702031602709
x =  11
9 -> 0.25196388261851016 0.006162528216704289 0.4473702031602709
x =  12
9 -> 0.23016930022573365 0.005948081264108352 0.4473702031602709
x =  13
9 -> 0.2112641083521445 0.006252821670428894 0.44737020

In [159]:
dssmid = 9
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (dssmid,)
    }


    topsize = 200
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[dssmid], f"{dssmid}"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
9 -> 0.5744469525959367 0.012392776523702033 0.5744469525959367
x =  1
9 -> 0.584018058690745 0.012020316027088036 0.5744469525959367
x =  2
9 -> 0.586410835214447 0.011653498871331828 0.5744469525959367
x =  3
9 -> 0.5699266365688487 0.012279909706546277 0.5744469525959367
x =  4
9 -> 0.5346388261851016 0.011822799097065465 0.5744469525959367
x =  5
9 -> 0.49005079006772007 0.012133182844243792 0.5744469525959367
x =  6
9 -> 0.44170993227990973 0.011918735891647854 0.5744469525959367
x =  7
9 -> 0.39646726862302484 0.012466139954853276 0.5744469525959367
x =  8
9 -> 0.35639954853273137 0.011952595936794583 0.5744469525959367
x =  9
9 -> 0.32224040632054174 0.011997742663656885 0.5744469525959367
x =  10
9 -> 0.2926354401805869 0.012042889390519187 0.5744469525959367
x =  11
9 -> 0.2685214446952596 0.01260722347629797 0.5744469525959367
x =  12
9 -> 0.2483803611738149 0.012641083521444697 0.5744469525959367
x =  13
9 -> 0.23149548532731376 0.012127539503386006 0.5744469525959367